In [1]:
from scipy.interpolate import interp1d
import polars as pl
import pandas as pd
import numpy as np
import json

In [ ]:
# Polars
df = pl.read_parquet('./data/pitching_data.parquet')
df.head()

In [ ]:
# Both
print("Loading in pitch marker mapping")
# Load in pitch_marker_mapping
with open("./data/pitch_marker_mapping.json", 'r') as f:
    pitch_marker_mapping = json.load(f)

# Polars
print("Converting pitch marker mapping into a Polars DataFrame")
# Unroll the JSON mapping into a DataFrame
mapping_records = []
for pitch_key, labels in pitch_marker_mapping.items():
    for marker_id, marker_label in enumerate(labels):
        mapping_records.append({
            "pitch_key": pitch_key,
            "markerID": marker_id,
            "markerLabel": marker_label
        })
mapping_df = pl.DataFrame(mapping_records)

In [ ]:
# Create a key to merge data with marker mapping
df = df.with_columns(
    (
    df['userID'].cast(pl.Utf8) + '_' +
    df['sessionID'].cast(pl.Utf8) + '_' +
    df['pitchNum'].cast(pl.Utf8)
    ).alias('pitch_key')
)

In [ ]:
print("Merging data with marker mapping")
# Merge data with marker mapping
df = df.join(
    mapping_df,
    on = ['pitch_key', 'markerID'],
    how = 'inner' # Only keeping markers that have a label
)

In [ ]:
print("Replacing markerID number with markerLabel description as markerID")
# Drop original ID, replace and rename with label
df = df.drop(['markerID'])
df = df.with_columns(
    df['markerLabel'].str.split('_')
    .list.get(-1)
    .alias('markerLabel')
)
df = df.rename({'markerLabel': 'markerID'})

In [ ]:
print("Loading in poi_metrics, getting unique session-key pitcher-handedness combinations")
# Loading points of interest data, which contains pitcher handedness
poi_metrics = pl.read_csv("./data/poi_metrics.csv")[["session_pitch", "session", "p_throws", "pitch_speed_mph"]]
poi_metrics = poi_metrics.with_columns(
    poi_metrics["session"].cast(pl.Utf8)
    .alias('session_key')
)
# Drop duplicates
poi_metrics_unique = poi_metrics.unique(subset = ['session_key', 'p_throws'])

In [ ]:
print("Merging marker data with handedness")
# Merging marker data with pitcher handedness data
df = df.with_columns(
    df['sessionID'].cast(pl.Utf8)
    .str.replace(r"^0+", "")
    .alias('session_key')
)
df = df.join(
    poi_metrics_unique,
    on = 'session_key',
    how = 'left'
)

In [ ]:
def swap_lr(label):
    if label.startswith("L"):
        return "R" + label[1:]
    if label.startswith("R"):
        return "L" + label[1:]
    return label

print("Mirroring left-handed pitchers about the y-axis")
# Mirror lefties about the y-axis -- Now every pitcher is treated as being the same handedness (right-handed)
mask = pl.col("p_throws") == "L"

df = df.with_columns([
    pl.when(mask)
      .then(pl.col("y") * -1)
      .otherwise(pl.col("y"))
      .alias("y"),

    pl.when(mask)
      .then(pl.col("markerID").map_elements(swap_lr,  return_dtype = pl.Utf8))
      .otherwise(pl.col("markerID"))
      .alias("markerID"),
])

In [ ]:
print("Removing markers that are not present in every pitch")
# Identify unique markers per pitch
pitch_marker_sets = (
    df.group_by(["userID", "sessionID", "pitchNum"]).agg(
        pl.col('markerID').unique()
    )
)
# All markers in original data
all_markers = set(df["markerID"].unique())

# Intersect all marker sets to find markers present in every pitch
common_markers = set.intersection(*[set(x) for x in pitch_marker_sets["markerID"]])

# Markers that were excluded
excluded_markers = all_markers - common_markers
print(f"Removed {excluded_markers} from marker set for consistency")

# Filter data to only include markers present in every pitch
df = df.filter(pl.col('markerID').is_in(list(common_markers)))

# Dropping redundant columns
df = df.drop(['session_key', 'pitch_key', 'session_pitch', 'session', 'pitch_speed_mph'])

print('Saving cleaned data to cleaned_data2.parquet')
df.write_parquet('./data/cleaned_data2.parquet')

In [2]:
data = pl.read_parquet('./data/cleaned_data2.parquet')
data.head()

userID,sessionID,height,weight,pitchNum,pitchType,pitchSpeed,frame,x,y,z,markerID,p_throws
str,str,str,str,str,str,f64,i64,f64,f64,f64,str,str
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.003564,-0.068032,1.581415,"""C7""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.057339,-0.186507,1.495219,"""CLAV""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.183473,0.22933,0.071697,"""LANK""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.217727,-0.112121,1.040045,"""LASI""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,-0.001984,-0.085127,1.731568,"""LBHD""","""R"""


In [3]:
import polars as pl
import numpy as np
from scipy.interpolate import interp1d

def normalize_frames_polars_interpolate(df: pl.DataFrame, N_TARGET_FRAMES: int = 101) -> pl.DataFrame:
    all_groups = []

    # Target normalized frame axis
    frame_norm = np.linspace(0, 100, N_TARGET_FRAMES)

    # Loop over each pitch explicitly
    for pitch_key, group in df.group_by(["userID", "sessionID", "pitchNum"]):
        # Process each marker separately
        for marker, marker_df in group.partition_by("markerID", as_dict=True).items():
            marker_df = marker_df.sort("frame")
            n_frames = marker_df.height
            if n_frames == 1:
                # single frame → replicate values
                interp_x = np.full(N_TARGET_FRAMES, marker_df["x"][0])
                interp_y = np.full(N_TARGET_FRAMES, marker_df["y"][0])
                interp_z = np.full(N_TARGET_FRAMES, marker_df["z"][0])
            else:
                # Original frame axis scaled 0–100
                orig_frames = np.linspace(0, 100, n_frames)
                interp_x = np.interp(frame_norm, orig_frames, marker_df["x"].to_numpy())
                interp_y = np.interp(frame_norm, orig_frames, marker_df["y"].to_numpy())
                interp_z = np.interp(frame_norm, orig_frames, marker_df["z"].to_numpy())

            # Build new DataFrame for this marker
            new_df = pl.DataFrame({
                "userID": pl.Series([marker_df["userID"][0]] * N_TARGET_FRAMES),
                "sessionID": pl.Series([marker_df["sessionID"][0]] * N_TARGET_FRAMES),
                "height": pl.Series([marker_df["height"][0]] * N_TARGET_FRAMES),
                "weight": pl.Series([marker_df["weight"][0]] * N_TARGET_FRAMES),
                "pitchNum": pl.Series([marker_df["pitchNum"][0]] * N_TARGET_FRAMES),
                "pitchType": pl.Series([marker_df["pitchType"][0]] * N_TARGET_FRAMES),
                "pitchSpeed": pl.Series([marker_df["pitchSpeed"][0]] * N_TARGET_FRAMES),
                "frame_norm": pl.Series(frame_norm).cast(pl.Int64),
                "x": pl.Series(interp_x),
                "y": pl.Series(interp_y),
                "z": pl.Series(interp_z),
                "markerID": pl.Series([marker][0] * N_TARGET_FRAMES),
                "p_throws": pl.Series([marker_df["p_throws"][0]] * N_TARGET_FRAMES),
            })
            all_groups.append(new_df)

    # Concatenate all markers and pitches
    return pl.concat(all_groups)


norm = normalize_frames_polars_interpolate(data)
norm.head()

userID,sessionID,height,weight,pitchNum,pitchType,pitchSpeed,frame_norm,x,y,z,markerID,p_throws
str,str,str,str,str,str,f64,i64,f64,f64,f64,str,str
"""000830""","""001615""","""70""","""193""","""006""","""FF""",84.4,0,0.083963,-0.195102,1.510169,"""C7""","""R"""
"""000830""","""001615""","""70""","""193""","""006""","""FF""",84.4,1,0.083345,-0.196894,1.510196,"""C7""","""R"""
"""000830""","""001615""","""70""","""193""","""006""","""FF""",84.4,2,0.082656,-0.198893,1.510414,"""C7""","""R"""
"""000830""","""001615""","""70""","""193""","""006""","""FF""",84.4,3,0.081439,-0.200518,1.511184,"""C7""","""R"""
"""000830""","""001615""","""70""","""193""","""006""","""FF""",84.4,4,0.079809,-0.2019,1.512036,"""C7""","""R"""


In [4]:
sample = norm.filter((pl.col('userID') == '000874') & (pl.col('sessionID') == '001562') & (pl.col('pitchNum') == '008'))
sample.head()

userID,sessionID,height,weight,pitchNum,pitchType,pitchSpeed,frame_norm,x,y,z,markerID,p_throws
str,str,str,str,str,str,f64,i64,f64,f64,f64,str,str
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,0,0.204906,0.081556,1.548954,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,1,0.204764,0.081695,1.548511,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,2,0.204347,0.081894,1.547612,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,3,0.203768,0.082146,1.545997,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,4,0.20277,0.082593,1.543732,"""C7""","""R"""


In [55]:
data.head()

userID,sessionID,height,weight,pitchNum,pitchType,pitchSpeed,frame,x,y,z,markerID,p_throws
str,str,str,str,str,str,f64,i64,f64,f64,f64,str,str
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.003564,-0.068032,1.581415,"""C7""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.057339,-0.186507,1.495219,"""CLAV""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.183473,0.22933,0.071697,"""LANK""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,0.217727,-0.112121,1.040045,"""LASI""","""R"""
"""000002""","""003034""","""73""","""207""","""002""","""FF""",80.9,0,-0.001984,-0.085127,1.731568,"""LBHD""","""R"""


In [5]:
sample = sample.rename({'frame_norm': 'frame'})
sample.head()


userID,sessionID,height,weight,pitchNum,pitchType,pitchSpeed,frame,x,y,z,markerID,p_throws
str,str,str,str,str,str,f64,i64,f64,f64,f64,str,str
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,0,0.204906,0.081556,1.548954,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,1,0.204764,0.081695,1.548511,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,2,0.204347,0.081894,1.547612,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,3,0.203768,0.082146,1.545997,"""C7""","""R"""
"""000874""","""001562""","""73""","""211""","""008""","""FF""",90.3,4,0.20277,0.082593,1.543732,"""C7""","""R"""


In [13]:
# Making Polars DataFrame where each row is a pitch (i.e. x_marker_frame, y_marker_frame, z_marker_frame are columns made from x, y, z, frame, and markerID columns of df)

def create_pitch_df(df):
    # Create a Polars DataFrame where each row is a pitch
    pitch_df = df.pivot(
        index = ["userID", "sessionID", "pitchNum", "height", "weight", "pitchType", "pitchSpeed", "p_throws"],
        on = ['markerID', 'frame_norm'],
        values = ['x', 'y', 'z']
    )
    
    # Change from naming convention of x_{"C7",0} to x_C7_0 
    for col in pitch_df.columns:
        if col.startswith('x_') or col.startswith('y_') or col.startswith('z_'):
            new_col_name = col[0] + '_' + col.split('{')[1].split(',')[0].split('"')[1] + '_' + col.split('{')[1].split(',')[1][:-1]
            pitch_df = pitch_df.rename({col: new_col_name})

    return pitch_df

pitch_df = create_pitch_df(norm)
pitch_df.head()

userID,sessionID,pitchNum,height,weight,pitchType,pitchSpeed,p_throws,x_C7_0,x_C7_1,x_C7_2,x_C7_3,x_C7_4,x_C7_5,x_C7_6,x_C7_7,x_C7_8,x_C7_9,x_C7_10,x_C7_11,x_C7_12,x_C7_13,x_C7_14,x_C7_15,x_C7_16,x_C7_17,x_C7_18,x_C7_19,x_C7_20,x_C7_21,x_C7_22,x_C7_23,x_C7_24,x_C7_25,x_C7_26,x_C7_27,x_C7_28,…,z_T10_64,z_T10_65,z_T10_66,z_T10_67,z_T10_68,z_T10_69,z_T10_70,z_T10_71,z_T10_72,z_T10_73,z_T10_74,z_T10_75,z_T10_76,z_T10_77,z_T10_78,z_T10_79,z_T10_80,z_T10_81,z_T10_82,z_T10_83,z_T10_84,z_T10_85,z_T10_86,z_T10_87,z_T10_88,z_T10_89,z_T10_90,z_T10_91,z_T10_92,z_T10_93,z_T10_94,z_T10_95,z_T10_96,z_T10_97,z_T10_98,z_T10_99,z_T10_100
str,str,str,str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""000830""","""001615""","""006""","""70""","""193""","""FF""",84.4,"""R""",0.083963,0.083345,0.082656,0.081439,0.079809,0.077851,0.075916,0.074271,0.072791,0.071628,0.070636,0.070077,0.070029,0.070497,0.071175,0.072377,0.07404,0.07606,0.078545,0.081175,0.0842,0.087706,0.091595,0.095516,0.099972,0.104498,0.109309,0.11467,0.120576,…,0.933641,0.929193,0.92388,0.918202,0.912281,0.907603,0.901959,0.894741,0.886316,0.876279,0.864821,0.855413,0.850368,0.848841,0.852427,0.859537,0.873523,0.896416,0.923476,0.94444,0.961891,0.977855,0.996491,1.015322,1.031601,1.045591,1.057143,1.067425,1.077119,1.086728,1.09632,1.105611,1.113023,1.122508,1.132782,1.14151,1.149718
"""001644""","""002926""","""002""","""67""","""178""","""FF""",84.1,"""L""",0.135224,0.134501,0.133416,0.131874,0.129822,0.127432,0.124325,0.121234,0.117683,0.114155,0.110527,0.106612,0.102823,0.099251,0.096161,0.093963,0.091929,0.09038,0.089476,0.089019,0.088939,0.089287,0.089976,0.090746,0.092056,0.093708,0.095369,0.09692,0.098935,…,0.853157,0.840363,0.827386,0.814102,0.801054,0.787645,0.773627,0.759056,0.743675,0.727938,0.712205,0.698287,0.685263,0.67459,0.665953,0.659308,0.658607,0.664736,0.673734,0.687631,0.706093,0.722222,0.737712,0.75388,0.774852,0.797169,0.823635,0.848112,0.869582,0.889087,0.904096,0.915687,0.925962,0.934665,0.940433,0.944627,0.949672
"""000610""","""001778""","""002""","""71""","""195""","""FF""",83.3,"""R""",0.201867,0.203744,0.205788,0.207914,0.210204,0.212612,0.215229,0.217911,0.220671,0.223412,0.226169,0.229045,0.232161,0.23504,0.238053,0.241067,0.244201,0.247668,0.250798,0.254093,0.257285,0.260533,0.264009,0.26749,0.271061,0.274583,0.278325,0.282105,0.286055,…,0.966357,0.965936,0.965177,0.964367,0.963232,0.960839,0.957105,0.950964,0.94349,0.934852,0.925962,0.916962,0.90909,0.902878,0.900622,0.901718,0.906336,0.914616,0.926323,0.940879,0.957888,0.977075,0.994276,1.009696,1.024956,1.038434,1.051144,1.064991,1.080331,1.093044,1.103269,1.113202,1.122126,1.128943,1.132536,1.135254,1.137682
"""000359""","""002734""","""004""","""72""","""194""","""FF""",88.4,"""R""",0.174219,0.174199,0.173987,0.173638,0.173065,0.172256,0.171157,0.169579,0.167756,0.165603,0.163166,0.160231,0.157191,0.153507,0.149657,0.14579,0.14147,0.137262,0.132967,0.129035,0.125271,0.121951,0.119349,0.117221,0.115761,0.114948,0.114707,0.115222,0.116058,…,1.031711,1.018685,1.005517,0.99209,0.979281,0.967918,0.956556,0.945483,0.933983,0.922893,0.911327,0.898025,0.882428,0.864545,0.8467,0.825884,0.804306,0.782254,0.764986,0.755435,0.75443,0.760088,0.76922,0.785189,0.797543,0.807747,0.823518,0.846381,0.865819,0.893286,0.923262,0.947932,0.965138,0.975521,0.983592,0.993493,1.005371
"""001706""","""003228""","""006""","""73""","""207""","""FF""",90.6,"""R""",0.153677,0.15364,0.153565,0.153484,0.153204,0.153293,0.153088,0.152109,0.152817,0.152525,0.152093,0.151884,0.151884,0.151481,0.151267,0.151123,0.150692,0.150111,0.149244,0.148548,0.147207,0.146443,0.144944,0.143384,0.142126,0.140522,0.139742,0.139526,0.140338,…,1.120632,1.105576,1.089798,1.075303,1.06087,1.047225,1.034017,1.022